## Aula 14: Criando e Versionando Aplicações LLM na Nuvem (100% Gratuito)

### a) Chamada da Dor
Você já passou horas refinando o "prompt perfeito" para o seu modelo de linguagem (LLM), apenas para sobrescrever o arquivo, alterar uma linha de código, e de repente... a IA começa a responder tudo errado? E pior: você não lembra qual era a versão anterior. Na consultoria de IA, perder a rastreabilidade de um prompt ou de uma configuração de modelo é o equivalente a jogar dinheiro do cliente no lixo. A dor de não saber "o que quebrou a aplicação" é real e custa caro.

### b) Introdução da Metodologia
A solução para isso é aplicar o **Versionamento de Código (Git)** aliado a um **Deploy em Nuvem Gratuita (Streamlit Community Cloud + GitHub)**. Não versionamos apenas o código, mas tratamos nossos *prompts* e *parâmetros do LLM* como código (Prompt-as-Code). Assim, qualquer alteração fica registrada, e o deploy é automatizado.

### c) História e Comparação
Num projeto recente de consultoria, uma equipe guardava seus prompts em um arquivo chamado `prompt_final_v4_agora_vai_mesmo.txt`. Um estagiário alterou uma instrução para "ser mais criativo" (aumentando a temperatura do modelo), o que fez um bot de atendimento financeiro começar a inventar taxas de juros. Sem versionamento, a equipe levou 2 dias para descobrir o erro.

**O Profissional Amador:** Salva scripts no Google Drive, roda localmente e manda `.zip` pro cliente.
**O Consultor Profissional:** Usa Git para rastrear cada vírgula do prompt, utiliza `.env` para proteger chaves de API e faz o deploy na nuvem onde o cliente acessa via link, com rollback instantâneo se algo der errado.

### d) Introdução Teórica com Dicas Práticas
Para termos uma aplicação em nuvem funcional e gratuita, precisamos de 3 pilares:
1. **O Motor (LLM):** Usaremos a API da OpenAI (modelos GPT), um dos padrões mais fortes do mercado para IA generativa.
2. **O Versionamento (GitHub):** Onde nosso código vai morar.
3. **O Frontend/Cloud (Streamlit Cloud):** Uma nuvem gratuita que lê o GitHub e transforma o Python em um site interativo.

💡 *Dica de Consultoria:* **NUNCA** coloque sua chave de API (`OPENAI_API_KEY`) diretamente no código. Se você subir isso para o GitHub, robôs roubarão sua chave em 5 segundos. Use variáveis de ambiente (Secrets).

### e) Implementação Simples
Vamos começar criando o núcleo da nossa aplicação: chamar a IA de forma segura.
*Nota: no Colab, usamos `userdata` para simular as variáveis de ambiente que estarão no `.env` local ou nos Secrets da Cloud.*

In [8]:
# Instalação da biblioteca oficial da OpenAI
!pip install -q -U openai

from openai import OpenAI
import os

# SIMULAÇÃO DE BOAS PRÁTICAS: Resgatando a chave de forma segura
# No mundo real (seu PC), você usaria a biblioteca python-dotenv e faria: os.getenv("OPENAI_API_KEY")
# Como estamos no Colab, se você não tiver a secret configurada na aba da chave à esquerda, coloque sua chave abaixo temporariamente APENAS PARA TESTE.

try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENAI_API_KEY')
except:
    API_KEY = "COLOQUE_SUA_CHAVE_AQUI" # Substitua pela sua chave da plataforma da OpenAI se não estiver usando Secrets

# 1. Configurar o Cliente da API
client = OpenAI(api_key=API_KEY)

# 2. Teste Rápido - Sem versionamento do prompt ainda
try:
    resposta = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {"role": "user", "content": "Qual é a regra de ouro para versionar código na nuvem?"}
        ]
    )
    print("Resposta da IA:\n", resposta.choices[0].message.content)
except Exception as e:
    print("Erro ao chamar a API da OpenAI. Verifique se sua chave possui saldo (quota). Erro detalhado:\n", e)

Resposta da IA:
 A regra de ouro para versionar código na nuvem é manter um controle rigoroso e eficiente do histórico de alterações do seu código-fonte. Aqui estão algumas diretrizes essenciais para seguir:

1. **Use um sistema de controle de versão**: Utilize ferramentas como Git, que permite rastrear mudanças, colaborar com outros desenvolvedores e reverter alterações, se necessário.

2. **Commits pequenos e frequentes**: Faça commits pequenos e significativos regularmente. Isso facilita a identificação de problemas e a compreensão do histórico de alterações.

3. **Mensagens de commit claras**: Escreva mensagens de commit descritivas que expliquem o que foi alterado e por quê. Isso ajuda a documentar o histórico do projeto.

4. **Estratégia de ramificação (branching)**: Use uma estratégia de ramificação eficaz, como Git Flow ou GitHub Flow, para gerenciar o desenvolvimento de novas funcionalidades, correção de bugs e lançamentos.

5. **Revisão de código**: Implemente práticas de rev

### f) Implementação Avançada: Preparando para a Nuvem com Versionamento de Prompts
Agora a prática de um consultor sênior. Vamos construir o código exatamente como ele deve ir para o seu arquivo `app.py` que será subido no GitHub e lido pelo Streamlit Cloud.

Aqui, nós **separamos o prompt das configurações do modelo**. Isso nos permite versionar (via Git) as mudanças de comportamento do bot.

In [9]:
%%writefile app.py
# A magic command acima (%%writefile) simula o versionamento salvando o código em um arquivo físico chamado app.py

import streamlit as st
from openai import OpenAI
import os

# === DICA DE CONSULTORIA: VERSIONAMENTO DE CONFIGURAÇÕES ===
# Em vez de espalhar configs pelo código, centralize.
# Num projeto real, isso poderia vir de um arquivo JSON (config_v1.json)
CONFIG_MODELO = {
    "temperature": 0.3, # Baixa temperatura para respostas mais precisas/profissionais
    "top_p": 0.9,
    "max_tokens": 1024,
}

# Versionamento do Prompt de Sistema (O cérebro do seu bot)
PROMPT_SISTEMA_V1 = """
Você é um assistente de engenharia de software sênior.
Sua função é explicar conceitos de Git e Cloud de forma resumida e direta.
Sempre termine a resposta com uma dica de ouro de consultoria.
"""

# === CONFIGURAÇÃO SEGURA DA NUVEM ===
# No Streamlit Cloud, configuramos as chaves na aba 'Secrets'. O Streamlit joga elas para o st.secrets.
try:
    API_KEY = st.secrets["OPENAI_API_KEY"]
except:
    # Fallback para ambiente de desenvolvimento local (usando variável de ambiente)
    API_KEY = os.getenv("OPENAI_API_KEY", "SUA_CHAVE_LOCAL_AQUI")

# Inicializando o cliente
client = OpenAI(api_key=API_KEY)

# === INTERFACE (FRONTEND) ===
st.set_page_config(page_title="Consultor Cloud AI", page_icon="☁️")
st.title("☁️ Consultor de Deploy & Versionamento")
st.write("Aplicação demonstrativa pronta para o Streamlit Community Cloud.")

user_input = st.text_input("Qual sua dúvida sobre deploy ou versionamento?")

if st.button("Perguntar à IA") and user_input:
    with st.spinner("Pensando como um sênior..."):
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": PROMPT_SISTEMA_V1},
                    {"role": "user", "content": user_input}
                ],
                temperature=CONFIG_MODELO["temperature"],
                top_p=CONFIG_MODELO["top_p"],
                max_tokens=CONFIG_MODELO["max_tokens"]
            )
            st.success("Resposta recebida:")
            st.write(response.choices[0].message.content)
        except Exception as e:
            st.error(f"Erro na comunicação com a IA: {e}")

# --- FIM DO ARQUIVO APP.PY ---

Overwriting app.py


### Passos Finais para o Deploy (Para você fazer pós-aula):
1. **Crie um repositório no GitHub** e faça o commit do arquivo `app.py` gerado acima.
2. Crie um arquivo `requirements.txt` com: `streamlit\nopenai` e faça o commit.
3. Vá para **share.streamlit.io**, conecte seu GitHub, selecione o repositório.
4. Em **Advanced Settings (Secrets)** na tela de deploy do Streamlit, adicione: `OPENAI_API_KEY = "sua_chave_real_aqui"`.
5. Clique em Deploy!

Pronto! Você tem uma aplicação rodando na nuvem, de graça, com código e prompt versionados profissionalmente.